# 04b — Cross-City Vocab Unification (multi-city)

Runs ONCE, after every city's `04_tvg_construction` has completed. Each
city built its own `highway_vocab`/`building_type_vocab` from its own
OSM extract (Step A of `04`, cached under `interim/osm_cache/{city}/`),
so the same integer currently means a DIFFERENT road/building type in
each city's saved `.pt` graphs. Pooling them as-is would silently
misalign the shared embedding tables in `06`/`07`.

**Revised for the multi-city, combined-directory setup:** every city's
TVG graphs already live together in ONE combined
`processed/tvg_graphs/` directory (city-prefixed filenames, e.g.
`bog_positive_123.pt`), not one directory per city like the old 2-city
pipeline. So this notebook no longer loops over separate per-city
`tvg_dir`s -- it remaps each city's OWN files (selected via `01`'s
`reconciled_points.parquet` `city` column, not a directory glob) into
ONE combined staging directory, then does a SINGLE combined swap at the
end, not one swap per city.

This notebook does NOT recompute isovist geometry or refetch OSM data.
It only: (1) unions every city's cached vocabulary into one, (2) remaps
the vocab-index tensors already saved inside every city's `.pt` TVG
graph, (3) writes the remapped graphs to a SEPARATE combined output
directory for you to spot-check before swapping them in, and (4)
updates each city's cached vocab JSON + `buildings.parquet` so any
future rerun (e.g. adding a fifth city) starts from the unified vocab
already.

Nothing here touches any city's raw source data or the already-
checkpointed per-point logs -- this only patches the categorical
indices inside the saved graph tensors.

In [ ]:
# ── Clone/update repo, mount Drive ──────────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric geopandas shapely pandas pyyaml

In [ ]:
# ── Load config, locate every city's cache + the ONE combined tvg_graphs dir ──
import yaml
from pathlib import Path
import pandas as pd

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

CITIES = paths_cfg["cities"]
INTERIM_DIR = Path(paths_cfg["interim_dir"])
PROCESSED_DIR = Path(paths_cfg["processed_dir"])

TVG_DIR = PROCESSED_DIR / "tvg_graphs"                    # ONE combined dir, every city
TVG_UNIFIED_DIR = PROCESSED_DIR / "tvg_graphs_unified"     # combined staging dir

reconciled = pd.read_parquet(INTERIM_DIR / "reconciled_points.parquet")
city_point_ids = {
    city: reconciled.loc[reconciled["city"] == city, "point_id"].tolist()
    for city in CITIES
}

for city in CITIES:
    cache_dir = INTERIM_DIR / "osm_cache" / city
    assert cache_dir.exists(), f"{city}: expected OSM cache at {cache_dir}"
    assert city_point_ids[city], f"{city}: no points found in reconciled_points.parquet"
assert TVG_DIR.exists(), f"expected combined TVG graphs at {TVG_DIR}"
print(f"Cities: {CITIES}")
for city in CITIES:
    print(f"  [{city}] {len(city_point_ids[city])} points")

In [ ]:
import vocab_merge as vm

hw_vocabs, bt_vocabs = {}, {}
for city in CITIES:
    cache_dir = INTERIM_DIR / "osm_cache" / city
    hw_vocabs[city] = vm.load_vocab(cache_dir / "highway_vocab.json")
    bt_vocabs[city] = vm.load_vocab(cache_dir / "building_type_vocab.json")
    print(f"[{city}] highway_vocab: {len(hw_vocabs[city])} types | building_type_vocab: {len(bt_vocabs[city])} types")

In [ ]:
# ── Build the unified vocab, across ALL cities ────────────────
unified_hw = vm.build_unified_vocab(*hw_vocabs.values(), fallback="unclassified")
unified_bt = vm.build_unified_vocab(*bt_vocabs.values(), fallback="unknown")

print(f"Unified highway_vocab: {len(unified_hw)} types (was {[len(v) for v in hw_vocabs.values()]} per city)")
print(f"Unified building_type_vocab: {len(unified_bt)} types (was {[len(v) for v in bt_vocabs.values()]} per city)")
print()
print("NOTE: any notebook building svg_kwargs/tvg_kwargs must read these sizes")
print("dynamically from the post-unification vocab JSON (see 06/07e/07g/07h's")
print("own pattern) -- NEVER hardcode a vocab size, it will silently be wrong")
print("the moment a city is added or removed.")

In [ ]:
# ── Remap each city's OWN files (by point_id, not a directory glob --
#    every city's graphs already share ONE combined tvg_graphs/ dir) into
#    a SEPARATE combined staging dir. Never overwrites tvg_graphs/ directly
#    -- spot-check the staging copy below, THEN run the swap cell at the end.
remap_stats = {}
for city in CITIES:
    n_ok, n_err = vm.remap_city_graphs(
        tvg_dir=TVG_DIR, out_dir=TVG_UNIFIED_DIR,
        old_highway_vocab=hw_vocabs[city], new_highway_vocab=unified_hw,
        old_building_vocab=bt_vocabs[city], new_building_vocab=unified_bt,
        point_ids=city_point_ids[city],
    )
    remap_stats[city] = (n_ok, n_err)

assert all(err == 0 for _, err in remap_stats.values()), "Fix errors above before continuing."

n_original = len(list(TVG_DIR.glob("*.pt")))
n_staged = len(list(TVG_UNIFIED_DIR.glob("*.pt")))
assert n_staged == n_original, (
    f"staging dir has {n_staged} files but the original has {n_original} -- "
    "swap_in_remapped_graphs would silently drop the difference. Investigate "
    "before continuing (likely a city whose point_ids didn't all have a TVG "
    "graph on disk yet)."
)
print(f"✅ Staged {n_staged}/{n_original} graphs across {len(CITIES)} cities.")

In [ ]:
# ── Spot-check: pick a few graphs per city, confirm old category string
#    still resolves to the SAME string under the new index ────────────
import random
import torch

inv_unified_hw = {i: c for c, i in unified_hw.items()}
inv_unified_bt = {i: c for c, i in unified_bt.items()}

for city in CITIES:
    inv_old_hw = {i: c for c, i in hw_vocabs[city].items()}
    inv_old_bt = {i: c for c, i in bt_vocabs[city].items()}

    sample_ids = random.sample(city_point_ids[city], min(3, len(city_point_ids[city])))
    print(f"\n=== {city} ===")
    for pid in sample_ids:
        old_data = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
        new_data = torch.load(TVG_UNIFIED_DIR / f"{pid}.pt", weights_only=False)

        old_hw_str = inv_old_hw[int(old_data['incident'].highway_type_idx.item())]
        new_hw_str = inv_unified_hw[int(new_data['incident'].highway_type_idx.item())]
        match = "✅" if old_hw_str == new_hw_str else "❌ MISMATCH"
        print(f"  {pid}: incident highway  old='{old_hw_str}' -> new='{new_hw_str}'  {match}")

        if old_data['building'].type_idx.numel():
            bi = 0
            old_bt_str = inv_old_bt[int(old_data['building'].type_idx[bi].item())]
            new_bt_str = inv_unified_bt[int(new_data['building'].type_idx[bi].item())]
            match = "✅" if old_bt_str == new_bt_str else "❌ MISMATCH"
            print(f"  {pid}: building[0] type  old='{old_bt_str}' -> new='{new_bt_str}'  {match}")

## Only run below once every spot-check above prints ✅

Swaps the unified graphs into `tvg_graphs/` (ONE combined swap, not one
per city), keeps the pre-unification originals safely at
`tvg_graphs_pre_unification_backup/`, and updates every city's cached
vocab JSON + `buildings.parquet` so this never needs re-running unless
another city is added later.

In [ ]:
vm.swap_in_remapped_graphs(TVG_DIR, TVG_UNIFIED_DIR)

for city in CITIES:
    cache_dir = INTERIM_DIR / "osm_cache" / city
    vm.save_vocab(unified_hw, cache_dir / "highway_vocab.json")
    vm.save_vocab(unified_bt, cache_dir / "building_type_vocab.json")
    vm.remap_buildings_parquet_type_idx(cache_dir / "buildings.parquet", bt_vocabs[city], unified_bt)
    print(f"[{city}] vocab cache + buildings.parquet updated to unified vocab.")

print(f"\nDone. All {len(CITIES)} cities' tvg_graphs/ now share one global vocab.")
print("Next: 05_dataset_assembly.ipynb")